# xFG upgrade: EDA

Shot quality with contest features at the release frame (`analytics/xfg`). Every table here is produced by the functions of `analytics/xfg/eda.py` on the synthetic season of `tests/_fx_xfg.py` (which prefers `analytics.fixtures` when it is importable).

**Fixture output, not a result.** Point `tables` at `validate.load_data_root(DATA_ROOT)` on the Mac to run the same cells on real data. Output is tables and markdown only: no plots, no raw data in the repository.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'analytics').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from analytics.xfg import eda as E, features as F, train as T, validate as V

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Inputs

A 6-game fixture season (24 possessions a game, every 3rd game held out). `validate.clean` runs the range, missing-value and duplicate checks and logs the rows in and out; `train.assemble` builds the feature frame (time-forward, `schema.assert_time_forward`) and sets the held-out rows aside before anything is fitted.

In [ ]:
tables = V.fixture_tables(seed=0, n_games=6, heldout_every=3, n_possessions=24)
tables, summaries = V.clean(tables)
V.summary_table(summaries)

In [ ]:
train, heldout = T.assemble(tables)
print('training attempts', len(train), 'from', train['sportvu_game'].nunique(), 'games;',
      'held-out attempts', len(heldout), 'from', sorted(heldout['sportvu_game'].unique()))
print('with contest features:', int(train['has_contest'].sum()), 'of', len(train))
train[F.KEYS + F.FEATURES].head()

## Make rate by distance bin (fixture output, not a result)

In [ ]:
E.make_rate_by_distance(train)

## Make rate by closest-defender bin (fixture output, not a result)

Attempts whose release frame was not localised (`has_contest` 0) sit in their own row.

In [ ]:
E.make_rate_by_defender(train)

In [ ]:
E.make_rate_by_zone(train)

## Feature distributions (fixture output, not a result)

In [ ]:
E.feature_distributions(train)

## Correlation table (Spearman, fixture output, not a result)

Descriptive only: the correlation with `made` is never used by the trainer.

In [ ]:
E.correlation_table(train)

## Leakage check: shuffled labels score at chance

`made` is permuted within each game and the three candidates are run out-of-fold (grouped folds by game, the same code path as `train.main`). A pipeline without leakage cannot beat the chance level (the entropy of the base rate for log loss, p(1 - p) for Brier). `at_chance` must be true for every candidate.

In [ ]:
check = E.shuffle_check(train, n_folds=3, seed=0)
assert check['at_chance'].all(), check
check

## Next

The candidate comparison, calibration tables and the held-out score are in `reports/xfg.md`, written by `python -m analytics.xfg.train --fixtures --report reports/xfg.md`. The real run on the Mac is the command in that report's last section.